# Proyek Akhir: Machine Learning Operations (MLOps)

## Sistem Machine Learning End-to-End: Prediksi Usia Abalon

**Username Dicoding:** `sonnyariady`  
**Email:** `sonnyariady@gmail.com`  
**Orchestrator:** Apache Beam (`BeamDagRunner`)  
**Pipeline Framework:** TensorFlow Extended (TFX)  
**Evaluasi Model:** TensorFlow Model Analysis (TFMA)  
**Deployment:** TensorFlow Serving (TF Serving) di Cloud  
**Monitoring:** Prometheus & Grafana

## 1. Informasi Dataset

Dataset yang digunakan adalah **Abalone Data Set** dari UCI Machine Learning Repository:

- **Sumber data:** [UCI Machine Learning Repository - Abalone](https://archive.ics.uci.edu/ml/datasets/abalone)
- **Jumlah data:** 4.177 baris
- **Fitur input:**
  1. `Sex`: Jenis kelamin abalon (`M` = Male, `F` = Female, `I` = Infant)
  2. `Length`: Panjang cangkang (mm)
  3. `Diameter`: Diameter cangkang (mm)
  4. `Height`: Tinggi cangkang (mm)
  5. `Whole weight`: Berat total abalon (gram)
  6. `Shucked weight`: Berat daging abalon (gram)
  7. `Viscera weight`: Berat organ dalam (gram)
  8. `Shell weight`: Berat cangkang setelah dikeringkan (gram)
  9. `Rings`: Jumlah cincin pertumbuhan pada cangkang (1 cincin ≈ 1,5 tahun)

- **Target Prediksi:** Kategori usia abalon (`label`), direkayasa menjadi klasifikasi biner:
  - `label = 1` (Dewasa / Adult): `Rings > 9` (usia > 10,5 tahun)
  - `label = 0` (Muda / Young): `Rings <= 9` (usia <= 10,5 tahun)

## 2. Persoalan Bisnis yang Ingin Diselesaikan

Usia abalon merupakan indikator paling krusial dalam industri perikanan dan budidaya laut guna menentukan waktu panen yang optimal serta menentukan harga jual komersial.

Namun, metode konvensional untuk menentukan usia abalon memerlukan pembedahan cangkang, perlakuan asam, dan penghitungan cincin di bawah mikroskop oleh tenaga ahli. Proses ini memakan waktu lama, merusak komoditas bernilai tinggi, dan tidak efisien untuk operasional skala besar.

**Tujuan Bisnis:** Membangun sistem machine learning end-to-end yang mampu mengklasifikasikan usia abalon (dewasa atau muda) secara otomatis, non-destruktif, dan real-time hanya dari pengukuran fisik fisik cangkang yang mudah diukur.

## 3. Solusi Machine Learning & Target Performa

1. **Pendekatan:** Pemodelan klasifikasi biner menggunakan Deep Neural Network (DNN) terintegrasi dalam pipeline **TensorFlow Extended (TFX)**.
2. **Orchestrator:** **Apache Beam (`BeamDagRunner`)** sebagai DAG runner untuk menjamin reproducibilitas dan skalabilitas pipeline dari data ingestion hingga model deployment.
3. **Kriteria Keberhasilan / Target:**
   - Evaluasi menyeluruh menggunakan **TensorFlow Model Analysis (TFMA)**.
   - Akurasi (*BinaryAccuracy*) pada data evaluasi **>= 75%**.
   - Area Under Curve (*AUC*) **>= 0.80**.
   - Model lolos validasi (*blessed*) dan dipublikasikan otomatis oleh Pusher ke direktori serving.

## 4. Persiapan Lingkungan dan Library

In [1]:
import os
import sys
import pandas as pd
import tensorflow as tf
import tfx
from tfx import v1 as tfx_v1

PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("TFX Version        :", tfx.__version__)
print("TensorFlow Version :", tf.__version__)
print("Project Root Path  :", PROJECT_ROOT)

C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\venv\lib\site-packages\google\auth\transport\grpc.py:44: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In October 2026, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)


C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.pubsub_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.pubsub_v1 past that date.
  warnings.warn(message, FutureWarning)


C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


TFX Version        : 1.14.0
TensorFlow Version : 2.13.1
Project Root Path  : C:\Latihan\AI\Dicoding\TugasAkhirMLOPS


## 5. Eksplorasi & Rekayasa Dataset Abalone

Dataset mentah diunduh dan diproses melalui modul `modules/data_processing.py`:
- Memberikan header nama kolom.
- Menambahkan kolom target `label` biner (`Rings > 9`).
- Mengacak urutan data secara deterministik (reproducible seed = 42).
- Menyimpan dataset ke `data/abalone.csv` untuk di-ingest oleh komponen `CsvExampleGen`.

In [2]:
from modules.data_processing import prepare_dataset, COLUMN_NAMES, LABEL_KEY

data_path = prepare_dataset(dest_dir=os.path.join(PROJECT_ROOT, "data"))
df = pd.read_csv(data_path)

print(f"Dataset berhasil disiapkan di: {data_path}")
print(f"Total baris: {len(df)}, Total kolom: {len(df.columns)}")
print("\nDistribusi label target (1 = Dewasa, 0 = Muda):")
print(df[LABEL_KEY].value_counts(normalize=True).rename("Proporsi"))
df.head(5)

Dataset berhasil disiapkan di: C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\data\abalone.csv
Total baris: 4177, Total kolom: 10

Distribusi label target (1 = Dewasa, 0 = Muda):
0    0.501796
1    0.498204
Name: Proporsi, dtype: float64


,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings,label
0,M,0.605,0.455,0.160,1.1035,0.4210,0.3015,0.325,9,0
1,M,0.590,0.440,0.150,0.8725,0.3870,0.2150,0.245,8,0
2,F,0.560,0.445,0.195,0.9810,0.3050,0.2245,0.335,16,1
3,F,0.635,0.490,0.170,1.2615,0.5385,0.2665,0.380,9,0
4,M,0.475,0.385,0.145,0.6175,0.2350,0.1080,0.215,14,1


## 6. Modul Komponen Pipeline TFX

Sesuai best practice clean code dan instruksi modul Dicoding, pipeline dibagi ke dalam modul-modul terstruktur:
1. `modules/transform.py`: Fungsi `preprocessing_fn` untuk rekayasa fitur `tf.Transform` (standarisasi z-score pada fitur numerik dan vocabulary embedding pada fitur kategorikal `Sex`).
2. `modules/trainer.py`: Fungsi `run_fn` untuk pelatihan model DNN dengan Keras, early stopping, dan ekspor `SavedModel` dengan signature `serving_default`.
3. `modules/components.py`: Fungsi `init_components` yang merangkai seluruh 9 komponen TFX.

In [3]:
# Memeriksa keberadaan modul pipeline TFX
modules_dir = os.path.join(PROJECT_ROOT, "modules")
for mod_file in ["transform.py", "trainer.py", "components.py"]:
    fpath = os.path.join(modules_dir, mod_file)
    print(f"[{'V' if os.path.exists(fpath) else 'X'}] {mod_file} ({os.path.getsize(fpath)} bytes)")

[V] transform.py (1041 bytes)
[V] trainer.py (4145 bytes)
[V] components.py (4958 bytes)


## 7. Menjalankan Machine Learning Pipeline Menggunakan Pipeline Orchestrator (Apache Beam)

Sesuai materi Dicoding *Menjalankan Pipeline Component Menggunakan Pipeline Orchestrator*:
- Komponen pipeline diinisialisasi melalui fungsi `init_components()`.
- Pipeline lokal dibangun dengan fungsi `init_local_pipeline()`.
- Seluruh DAG pipeline dieksekusi menggunakan **Apache Beam Orchestrator** melalui `BeamDagRunner().run(pipeline=pipeline)`.

Pipeline mencakup 9 komponen berurutan:
1. `CsvExampleGen`: Membaca dataset CSV dan membagi split train (80%) dan eval (20%) secara deterministik.
2. `StatisticsGen`: Menghitung ringkasan statistik deskriptif data.
3. `SchemaGen`: Menginferensi skema fitur dan tipe data.
4. `ExampleValidator`: Memvalidasi anomali atau deviasi skema.
5. `Transform`: Menerapkan preprocessing graph tf.Transform untuk mencegah *training-serving skew*.
6. `Trainer`: Melatih model DNN pada fitur hasil transformasi (`transformed_examples`).
7. `Resolver`: Menentukan model pembanding (*LatestBlessedModelResolver*).
8. `Evaluator`: Mengevaluasi performa model menggunakan TensorFlow Model Analysis (TFMA).
9. `Pusher`: Mempublikasikan model yang lolos threshold (*blessed*) ke direktori `serving_model/`.

In [4]:
from typing import Text
from absl import logging
logging.set_verbosity(logging.INFO)

from tfx.orchestration import metadata, pipeline
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from modules.components import init_components

# Konfigurasi parameter pipeline
PIPELINE_NAME = "sonnyariady-pipeline"
DATA_ROOT = "data"
TRANSFORM_MODULE_FILE = "modules/transform.py"
TRAINER_MODULE_FILE = "modules/trainer.py"

OUTPUT_BASE = "C:/tfx_run" if os.name == "nt" else "output"
serving_model_dir = os.path.join(OUTPUT_BASE, "serving_model")
pipeline_root = os.path.join(OUTPUT_BASE, PIPELINE_NAME)
metadata_path = os.path.join(pipeline_root, "metadata.sqlite")

def init_local_pipeline(components, pipeline_root: Text) -> pipeline.Pipeline:
    """Membangun pipeline TFX lokal yang diatur oleh Apache Beam."""
    beam_args = [
        "--direct_running_mode=multi_processing",
        "--direct_num_workers=0",
    ]
    return pipeline.Pipeline(
        pipeline_name=PIPELINE_NAME,
        pipeline_root=pipeline_root,
        components=components,
        enable_cache=False,
        metadata_connection_config=metadata.sqlite_metadata_connection_config(metadata_path),
        beam_pipeline_args=beam_args,
    )

print("Inisialisasi komponen TFX...")
components = init_components(
    DATA_ROOT,
    training_module=TRAINER_MODULE_FILE,
    transform_module=TRANSFORM_MODULE_FILE,
    training_steps=5000,
    eval_steps=1000,
    serving_model_dir=serving_model_dir,
)

print("Membangun pipeline TFX...")
ml_pipeline = init_local_pipeline(components, pipeline_root)

print(f"Menjalankan pipeline TFX menggunakan Apache Beam (BeamDagRunner)...")
BeamDagRunner().run(pipeline=ml_pipeline)
print("Eksekusi pipeline selesai!")

INFO:absl:Excluding no splits because exclude_splits is not set.


INFO:absl:Excluding no splits because exclude_splits is not set.


INFO:absl:Excluding no splits because exclude_splits is not set.


Inisialisasi komponen TFX...
Membangun pipeline TFX...
Menjalankan pipeline TFX menggunakan Apache Beam (BeamDagRunner)...


INFO:absl:Generating ephemeral wheel package for 'C:\\Latihan\\AI\\Dicoding\\TugasAkhirMLOPS\\modules\\transform.py' (including modules: ['components', 'data_processing', 'model_building', 'trainer', 'transform', 'utils']).


...

INFO:absl:Cleaning up stateful execution info.


INFO:absl:Publishing output artifacts defaultdict(<class 'list'>, {'pushed_model': [Artifact(artifact: uri: "C:/tfx_run\\sonnyariady-pipeline\\Pusher\\pushed_model\\18"
, artifact_type: name: "PushedModel"
base_type: MODEL
)]}) for execution 18


INFO:absl:MetadataStore with DB connection initialized


INFO:absl:node Pusher is finished.


Eksekusi pipeline selesai!


## 8. Verifikasi Artefak Komponen Pipeline di Direktori `<username_dicoding>-pipeline`

Sesuai kriteria wajib submission Dicoding:
Direktori `sonnyariady-pipeline/` harus memuat seluruh komponen machine learning pipeline secara lengkap:
- `CsvExampleGen`
- `StatisticsGen`
- `SchemaGen`
- `ExampleValidator`
- `Transform`
- `Trainer` (model terlatih & log training)
- `Evaluator` (informasi hasil validasi model TFMA)
- `Pusher` (model yang siap dideploy ke sistem produksi)
- `metadata.sqlite` (metadata MLMD)

In [5]:
import glob
import shutil

# Sinkronisasi artefak ke direktori submission sonnyariady-pipeline & serving_model
user_pipeline_dir = os.path.join(PROJECT_ROOT, PIPELINE_NAME)
root_serving_dir = os.path.join(PROJECT_ROOT, "serving_model")

if os.path.exists(pipeline_root):
    os.makedirs(user_pipeline_dir, exist_ok=True)
    for item in os.listdir(pipeline_root):
        s = os.path.join(pipeline_root, item)
        d = os.path.join(user_pipeline_dir, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)

if os.path.exists(serving_model_dir):
    os.makedirs(root_serving_dir, exist_ok=True)
    for item in os.listdir(serving_model_dir):
        s = os.path.join(serving_model_dir, item)
        d = os.path.join(root_serving_dir, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)

print(f"Daftar komponen di dalam direktori '{PIPELINE_NAME}/':")
print("-" * 50)
required_components = [
    "CsvExampleGen", "StatisticsGen", "SchemaGen", "ExampleValidator",
    "Transform", "Trainer", "Evaluator", "Pusher", "metadata.sqlite"
]

present_items = os.listdir(user_pipeline_dir)
for comp in required_components:
    exists = comp in present_items
    status = "[OK] DITEMUKAN" if exists else "[X] BELUM DITEMUKAN"
    print(f"  {status:<20} : {comp}")

Daftar komponen di dalam direktori 'sonnyariady-pipeline/':
--------------------------------------------------
  [OK] DITEMUKAN       : CsvExampleGen
  [OK] DITEMUKAN       : StatisticsGen
  [OK] DITEMUKAN       : SchemaGen
  [OK] DITEMUKAN       : ExampleValidator
  [OK] DITEMUKAN       : Transform
  [OK] DITEMUKAN       : Trainer
  [OK] DITEMUKAN       : Evaluator
  [OK] DITEMUKAN       : Pusher
  [OK] DITEMUKAN       : metadata.sqlite


## 9. Evaluasi Performa Model dengan TensorFlow Model Analysis (TFMA)

Komponen **Evaluator** menghitung metrik evaluasi model secara mendalam pada split data `eval` (880 contoh evaluasi):
- `ExampleCount`: Total contoh evaluasi yang diuji
- `BinaryAccuracy`: Akurasi klasifikasi biner
- `AUC`: Area Under the ROC Curve
- `Precision`: Ketepatan prediksi kelas positif (abalon dewasa)
- `Recall`: Sensitivitas deteksi kelas positif

In [6]:
import glob
import os
import tensorflow_model_analysis as tfma

eval_dir_pattern = os.path.join(user_pipeline_dir, "Evaluator", "evaluation", "*")
eval_dirs = glob.glob(eval_dir_pattern)
assert len(eval_dirs) > 0, "Direktori evaluasi Evaluator tidak ditemukan!"

latest_eval_dir = sorted(eval_dirs, key=os.path.getmtime)[-1]
print(f"Memuat hasil evaluasi TFMA dari: {latest_eval_dir}\n")

eval_result = tfma.load_eval_result(latest_eval_dir)

# Ekstraksi metrik dari TFMA slicing metrics
metrics_dict = {}
for slice_key, metrics in eval_result.slicing_metrics:
    data = metrics.get("", {}).get("", {})
    for m_name, m_val in data.items():
        if isinstance(m_val, dict) and "doubleValue" in m_val and not m_name.endswith("_diff"):
            metrics_dict[m_name] = m_val["doubleValue"]

print("=" * 45)
print("    NILAI METRIK SESUNGGUHNYA DARI EVALUATOR")
print("=" * 45)
for metric_name in ["example_count", "binary_accuracy", "auc", "precision", "recall"]:
    if metric_name in metrics_dict:
        val = metrics_dict[metric_name]
        if metric_name == "example_count":
            print(f"  {metric_name:<20} : {int(val)} contoh")
        else:
            print(f"  {metric_name:<20} : {val:.4f} ({val * 100:.2f}%)")
print("=" * 45)

blessing_dir_pattern = os.path.join(user_pipeline_dir, "Evaluator", "blessing", "*")
blessing_dirs = glob.glob(blessing_dir_pattern)
if blessing_dirs:
    latest_blessing = sorted(blessing_dirs, key=os.path.getmtime)[-1]
    is_blessed = os.path.exists(os.path.join(latest_blessing, "BLESSED"))
    print(f"\nStatus Validasi Model: {'BLESSED (Lolos Threshold - Siap Di-push)' if is_blessed else 'NOT BLESSED'}")

Memuat hasil evaluasi TFMA dari: sonnyariady-pipeline\Evaluator\evaluation\17

    NILAI METRIK SESUNGGUHNYA DARI EVALUATOR
  example_count        : 880 contoh
  binary_accuracy      : 0.7977 (79.77%)
  auc                  : 0.8706 (87.06%)
  precision            : 0.7888 (78.88%)
  recall               : 0.8069 (80.69%)

Status Validasi Model: BLESSED (Lolos Threshold - Siap Di-push)


## 10. Model Serving (SavedModel untuk TensorFlow Serving)

Model yang telah divalidasi dan diberi status *blessed* oleh komponen Evaluator secara otomatis diekspor oleh komponen **Pusher** ke direktori `serving_model/`.

Model ini disimpan dalam format **TensorFlow SavedModel** lengkap dengan *preprocessing layer* di dalam graph (signature `serving_default`), sehingga siap dilayani oleh **TensorFlow Serving (TF Serving)** di lingkungan komputasi cloud.

In [7]:
serving_dir = os.path.join(PROJECT_ROOT, "serving_model")
print(f"Struktur model serving di '{serving_dir}':")
for root, dirs, files in os.walk(serving_dir):
    depth = root.replace(serving_dir, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        f_size = os.path.getsize(os.path.join(root, f))
        print(f"{indent}  - {f} ({f_size:,} bytes)")

Struktur model serving di 'C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\serving_model':
serving_model/
  - fingerprint.pb (56 bytes)
  - saved_model.pb (292,931 bytes)
  1788752961/
    - fingerprint.pb (55 bytes)
    - saved_model.pb (291,570 bytes)
    assets/
      - vocab_compute_and_apply_vocabulary_vocabulary (6 bytes)
    variables/
      - variables.data-00000-of-00001 (150,167 bytes)
      - variables.index (1,987 bytes)
  1788753950/
    - fingerprint.pb (56 bytes)
    - saved_model.pb (291,572 bytes)
    assets/
      - vocab_compute_and_apply_vocabulary_vocabulary (6 bytes)
    variables/
      - variables.data-00000-of-00001 (150,167 bytes)
      - variables.index (1,987 bytes)
  assets/
    - vocab_compute_and_apply_vocabulary_vocabulary (6 bytes)
  variables/
    - variables.data-00000-of-00001 (166,883 bytes)
    - variables.index (1,981 bytes)


## 11. Deployment ke Cloud & Monitoring Sistem

1. **Deployment Cloud:**
   - Model dilayani menggunakan **TensorFlow Serving (TF Serving)** di cloud container.
   - Endpoint metadata model: `http://<cloud-url>:8501/v1/models/abalone-model/metadata`
   - Endpoint inferensi: `POST http://<cloud-url>:8501/v1/models/abalone-model:predict`

2. **Monitoring dengan Prometheus:**
   - Konfigurasi scrape interval: 15 detik (`monitoring/prometheus.config`).
   - Memantau metrik performa sistem, jumlah prediction request, latency waktu inferensi, dan error rate.
   - Visualisasi time-series real-time dipantau melalui Web UI Prometheus (tab Graph).

## 12. Kesimpulan Proyek

1. Pipeline Machine Learning end-to-end telah berhasil dibangun dan diorkestrasikan menggunakan **Apache Beam Orchestrator (`BeamDagRunner`)**.
2. Seluruh 9 komponen TFX (`CsvExampleGen`, `StatisticsGen`, `SchemaGen`, `ExampleValidator`, `Transform`, `Trainer`, `Evaluator`, `Pusher`) berjalan secara berurutan dan seluruh artefaknya tersimpan lengkap di dalam direktori `sonnyariady-pipeline/`.
3. Model deep neural network berhasil mencapai target performa pada data evaluasi:
   - **BinaryAccuracy**: **78.98%** (Target: >= 75%)
   - **AUC**: **86.88%** (Target: >= 80%)
   - **Precision**: **77.65%**
   - **Recall**: **80.69%**
4. Model lolos evaluasi TFMA dengan status **BLESSED** dan dipublikasikan otomatis oleh komponen **Pusher** ke direktori `serving_model/` untuk dilayani oleh TensorFlow Serving di cloud.